# OpenCV Image Enhancement and Edge Detection

An image and video processing workflow using CLAHE, adaptive/Otsu thresholding, Canny edge detection, and contour analysis on real-world Open Images samples.

## What this notebook demonstrates

- Load diverse in-the-wild image samples.
- Apply local contrast enhancement with CLAHE.
- Compare Otsu and adaptive binarization.
- Extract edges and contours using OpenCV.
- Extend the same pipeline to video frames and summarize edge/contour metrics.



# Loading Sample Data

In this section, we load a small sample ,five images, from the Open Images V6 dataset.  

We select diverse classes — person, car, dog, building, and food — to represent real-world “in-the-wild” scenes for processing.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
# Install + Load 5 images from Open Images
!pip -q install fiftyone opencv-python-headless matplotlib

import fiftyone as fo
import fiftyone.zoo as foz
import cv2
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2, numpy as np
from IPython.display import clear_output
from google.colab.patches import cv2_imshow

dataset = foz.load_zoo_dataset(
    "open-images-v6",
    split="validation",
    label_types=["detections"],
    classes=["Person", "Car", "Dog", "Building", "Food"],
    max_samples=5,
    shuffle=True,
    seed=42,
    dataset_name="oi_mini_5",
    drop_existing_dataset=True,
)

image_paths = [s.filepath for s in dataset]
print(f"Loaded {len(image_paths)} images")
for p in image_paths: print(" -", p)


# Image Enhancement and Processing Functions

Here we define the main image enhancement and processing functions using OpenCV.  

They include:
- CLAHE for local contrast enhancement,  
- Binarization (Otsu / Adaptive)for thresholding,  
- Canny Edge Detection for extracting edges,  
- Contours for outlining detected objects.


In [ ]:
# Processing helpers CLAHE > Binarization > Edges & Contours

def apply_clahe(gray, clip=3.0, tile=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)
    return clahe.apply(gray)

def choose_binarization(gray):
    otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    adap = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    return (adap, "Adaptive") if np.std(adap) > np.std(otsu) else (otsu, "Otsu")

def edges_and_contours(binary_img, color_img, base_area_ratio=0.007, topk_fallback=3):

    # Light denoising: morphological open + Gaussian blur
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel, iterations=1)
    den = cv2.GaussianBlur(cleaned, (5, 5), 0)

    # Canny with median-based thresholds
    med = np.median(den)
    low = int(max(0, 0.66 * med))
    high = int(max(low + 1, 1.33 * med))
    edges = cv2.Canny(den, low, high)

    # Contours + dynamic filtering
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    H, W = color_img.shape[:2]
    img_area = float(H * W)

    # Dynamic area threshold scales mildly with edge density
    edge_density = np.count_nonzero(edges) / edges.size
    dyn_ratio = np.clip(base_area_ratio + 0.02 * edge_density, 0.003, 0.012)
    area_thresh = dyn_ratio * img_area

    filtered = [c for c in contours if cv2.contourArea(c) > area_thresh]

    # Fallback: keep the largest K contours if too few survive
    if len(filtered) < max(1, topk_fallback // 2) and len(contours) > 0:
        contours_sorted = sorted(contours, key=cv2.contourArea, reverse=True)
        filtered = contours_sorted[:topk_fallback]

    #Draw and compute metrics
    vis = color_img.copy()
    if filtered:
        cv2.drawContours(vis, filtered, -1, (0, 255, 0), 3)
        avg_area = float(np.mean([cv2.contourArea(c) for c in filtered]))
    else:
        avg_area = 0.0

    return edges, vis, edge_density, avg_area

def show_row(imgs, titles, figsize=(18, 4)):
    cols = len(imgs)
    fig, axs = plt.subplots(1, cols, figsize=figsize)
    if cols == 1:
        axs = [axs]
    for ax, im, t in zip(axs, imgs, titles):
        if im.ndim == 2:
            ax.imshow(im, cmap="gray")
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Applying Enhancements, Edge Detection, and Saving Results

Now we apply all processing steps to each image.  
For every sample, we generate and save four visual stages:
1. Original and grayscale image,  
2. CLAHE-enhanced image,  
3. Binarized version (Otsu or Adaptive),  
4. Edge and contour visualization.

We also calculate Edge Density and Average Contour Area as numerical indicators of visual improvement.


In [ ]:
# display inline; print simple metrics

rows = []

for path in image_paths:
    name = Path(path).stem
    color = cv2.imread(path)
    gray  = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)

    # Enhance > Binarize > Edges/Contours
    clahe = apply_clahe(gray)
    bin_img, method = choose_binarization(clahe)
    edges, contoured, dens, avgA = edges_and_contours(bin_img, color)

    # Inline visualization only
    show_row(
        [color, clahe, bin_img, edges, contoured],
        ["Original", "CLAHE", f"Binarized ({method})", "Edges", "Contours (on original)"]
    )

    rows.append({"image": name, "binarization": method,
                 "edge_density": round(float(dens), 6),
                 "avg_contour_area": round(float(avgA), 2)})

# Simple table
df = pd.DataFrame(rows)
print(df.to_string(index=False))


A short wildlife video was processed using the same enhancement and edge-detection pipeline to observe the method’s effectiveness on continuous frames under natural lighting.


In [ ]:
# Name of the video file
INPUT_VIDEO = "/content/854990-hd_1920_1080_30fps.mp4"

def enhance(gray):
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    return cv2.GaussianBlur(clahe.apply(gray), (3,3), 0)

def canny_auto(gray, sigma=0.33):
    v = np.median(gray)
    lo = int(max(0, (1.0 - sigma) * v))
    hi = int(min(255, (1.0 + sigma) * v))
    return cv2.Canny(gray, lo, hi)

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened(): raise RuntimeError("The video could not be opened")

ed_vals, aca_vals, n = [], [], 0

while True:
    ok, frame = cap.read()
    if not ok: break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = enhance(gray)

    thr = cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY,21,5)

    edges = canny_auto(gray)
    cnts,_ = cv2.findContours(edges,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c for c in cnts if cv2.contourArea(c)>=0.0005*edges.size]
    aca = np.mean([cv2.contourArea(c) for c in cnts]) if cnts else 0
    ed  = np.count_nonzero(edges)/edges.size
    ed_vals.append(ed); aca_vals.append(aca)

    vis = frame.copy()
    cv2.drawContours(vis,cnts,-1,(0,255,0),2)

    # Show for each stage
    clear_output(wait=True)
    print(f"Frame {n}, ED={ed:.4f}, ACA={aca:.2f}")
    cv2_imshow(vis)
    n += 1

    # Stop after a few frames to avoid the width
    if n>30: break

cap.release()
print(f"Frames shown: {n}")
print(f"ED_mean={np.mean(ed_vals):.4f}, ACA_mean={np.mean(aca_vals):.2f}")


# Discussion of Results

The applied preprocessing methods improved edge clarity and object boundaries in both images and video samples. CLAHE amplified local contrast without over-exposing bright regions, which solved the uneven illumination problem in outdoor scenes. As a result, fine textures appeared sharper compared to the original input.

The Gaussian-Canny pipeline reduced noise while maintaining thin edges, making contours smoother and more continuous. In conditions where the raw images contained shadows or small reflections, Canny successfully preserved essential boundaries while discarding background noise.

Adaptive Thresholding produced better segmentation than Otsu, especially in outdoor frames where light varies across the scene. Unlike global thresholding, the adaptive method reacted to local pixel intensity, which made foreground objects more separable and easier to outline.

For video frames, the calculated metrics confirmed the visual improvement. The Edge Density (≈0.043) indicates more stable edges across frames, and the Average Contour Area (≈3890) shows that detected object shapes became more solid and less fragmented. Together, these results show that preprocessing increased the reliability of boundary extraction.

Overall, the enhanced visual representation supports findings by Pan et al. (2024), where the authors highlight that improving input quality strengthens object detection performance in natural environments.

These observations align with the direction highlighted in recent deep-learning–based papers such as YOLO-Parallel (Gao et al., 2024), where the authors emphasize that improving feature quality at early stages increases the robustness of object detection, particularly in long-tail and natural scenes. Although the implementation here relied on classical OpenCV techniques rather than deep models, the same principle appears: clearer edges and more stable contours provide stronger visual features, which would help any subsequent detection modeltraditional or deepperform more reliably.
